# IBM Quantum Hardware — Live Demonstrations

---

## Run Quantum Circuits on REAL Quantum Computers

This notebook connects to **IBM Quantum hardware** via the cloud and runs quantum circuits on real superconducting qubit processors. See how noise, decoherence, and real-world physics affect quantum computation compared to perfect simulators.

### What We'll Run on Real Hardware

| Demo | Qubits | Concept |
|------|--------|---------|
| 1. Single Qubit | 1 | Superposition & measurement statistics |
| 2. Bell State | 2 | Entanglement on real qubits |
| 3. GHZ State | 3-5 | Multi-qubit entanglement fidelity |
| 4. Quantum Random Number Generator | 4 | True randomness from hardware |
| 5. Grover's Search (2-qubit) | 2 | Algorithm on real device |
| 6. Noise Comparison | 2 | Simulator vs real hardware side-by-side |

### Setup

1. Get your free API token at https://quantum.ibm.com/account
2. Paste it in the cell below
3. Run all cells!

In [ ]:
# ============================================
# SETUP: Install & Configure IBM Quantum
# ============================================

# Uncomment to install if needed:
# !pip install qiskit qiskit-ibm-runtime qiskit-aer

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Qiskit imported successfully!')

In [ ]:
# ============================================
# PASTE YOUR IBM QUANTUM TOKEN BELOW
# ============================================
#
# Get it from: https://quantum.ibm.com/account
# (Sign in -> Profile icon -> Account settings -> API token -> Copy)
#
# FREE tier gives you access to real quantum computers!

IBM_TOKEN = "YOUR_TOKEN_HERE"  # <-- Paste your token inside the quotes

# ============================================
# Connect to IBM Quantum
# ============================================

USE_REAL_HARDWARE = False  # Will be set to True if connection succeeds
backend = None
service = None

if IBM_TOKEN != "YOUR_TOKEN_HERE" and len(IBM_TOKEN) > 20:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
        
        # Save account (only needs to run once)
        QiskitRuntimeService.save_account(
            channel="ibm_quantum",
            token=IBM_TOKEN,
            overwrite=True
        )
        
        # Connect
        service = QiskitRuntimeService(channel="ibm_quantum")
        
        # Get least busy backend
        backends = service.backends(
            simulator=False,
            operational=True,
            min_num_qubits=5
        )
        
        if backends:
            # Sort by queue length
            backend = service.least_busy(
                simulator=False,
                operational=True,
                min_num_qubits=5
            )
            USE_REAL_HARDWARE = True
            print(f'Connected to IBM Quantum!')
            print(f'Backend: {backend.name}')
            print(f'Qubits: {backend.num_qubits}')
            print(f'Status: {backend.status().pending_jobs} jobs in queue')
            print(f'\nYou are running on a REAL quantum computer!')
        else:
            print('No available backends. Using simulator.')
    except Exception as e:
        print(f'Could not connect to IBM Quantum: {e}')
        print('Falling back to local simulator.')
else:
    print('No IBM token provided. Using local AerSimulator.')
    print('To use real hardware, paste your token above.')
    print('Get one free at: https://quantum.ibm.com/account')

# Fallback simulator
sim_backend = AerSimulator()
print(f'\nSimulator ready: AerSimulator')
print(f'Hardware mode: {"ON - Real Quantum Computer" if USE_REAL_HARDWARE else "OFF - Simulator Only"}')

In [ ]:
# ============================================
# HELPER: Run circuit on hardware + simulator
# ============================================

def run_circuit(qc, shots=4096, label='Circuit'):
    """
    Run a circuit on both simulator and real hardware (if available).
    Returns dict with 'simulator' and optionally 'hardware' counts.
    """
    results = {}
    
    # Always run on simulator
    qc_sim = qc.copy()
    qc_sim.measure_all() if not any(isinstance(inst.operation, type(qc_sim.measure(0, 0).data[-1].operation)) for inst in qc_sim.data if hasattr(inst, 'operation')) else None
    sim_job = sim_backend.run(transpile(qc, sim_backend), shots=shots)
    results['simulator'] = dict(sim_job.result().get_counts())
    print(f'[Simulator] {label} complete ({shots} shots)')
    
    # Run on real hardware if available
    if USE_REAL_HARDWARE and backend is not None:
        try:
            from qiskit_ibm_runtime import SamplerV2
            qc_hw = transpile(qc, backend, optimization_level=3)
            sampler = SamplerV2(backend)
            job = sampler.run([qc_hw], shots=shots)
            print(f'[Hardware] Job submitted to {backend.name}...')
            result = job.result()
            # Extract counts from SamplerV2 result
            pub_result = result[0]
            counts_raw = pub_result.data.meas.get_counts()
            results['hardware'] = dict(counts_raw)
            print(f'[Hardware] {label} complete!')
        except Exception as e:
            print(f'[Hardware] Error: {e}')
            print('[Hardware] Showing simulator results only.')
    
    return results

def plot_comparison(results, title, expected=None):
    """
    Plot simulator vs hardware results side by side.
    """
    has_hw = 'hardware' in results
    n_plots = 3 if (has_hw and expected) else (2 if (has_hw or expected) else 1)
    
    fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    plot_idx = 0
    
    # Expected (ideal)
    if expected:
        ax = axes[plot_idx]; plot_idx += 1
        states = sorted(expected.keys())
        total = sum(expected.values())
        probs = [expected.get(s, 0)/total for s in states]
        ax.bar(states, probs, color='#4CAF50', alpha=0.8, edgecolor='black')
        ax.set_title('Ideal (Theory)', fontsize=13, fontweight='bold')
        ax.set_ylabel('Probability'); ax.set_ylim(0, 1.1)
        ax.grid(True, alpha=0.3, axis='y')
    
    # Simulator
    ax = axes[plot_idx]; plot_idx += 1
    sim_counts = results['simulator']
    all_states = sorted(set(list(sim_counts.keys()) + (list(results.get('hardware', {}).keys()))))
    if not all_states:
        all_states = sorted(sim_counts.keys())
    total_sim = sum(sim_counts.values())
    sim_probs = [sim_counts.get(s, 0)/total_sim for s in all_states]
    ax.bar(all_states, sim_probs, color='#2196F3', alpha=0.8, edgecolor='black')
    ax.set_title('Simulator (Noiseless)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Probability'); ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Hardware
    if has_hw:
        ax = axes[plot_idx]
        hw_counts = results['hardware']
        total_hw = sum(hw_counts.values())
        hw_probs = [hw_counts.get(s, 0)/total_hw for s in all_states]
        ax.bar(all_states, hw_probs, color='#FF5722', alpha=0.8, edgecolor='black')
        ax.set_title(f'Real Hardware ({backend.name})', fontsize=13, fontweight='bold')
        ax.set_ylabel('Probability'); ax.set_ylim(0, 1.1)
        ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle(title, fontsize=15, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()

print('Helper functions ready!')

---

## Demo 1: Superposition — Single Qubit

The simplest quantum experiment: apply Hadamard to $|0\rangle$ and measure.

$$H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$$

**Ideal**: exactly 50/50. **Real hardware**: slight bias due to readout errors and decoherence.

In [ ]:
# Demo 1: Superposition

qc1 = QuantumCircuit(1, 1)
qc1.h(0)
qc1.measure(0, 0)

print('Circuit:')
print(qc1.draw(output='text'))
print()

results1 = run_circuit(qc1, shots=4096, label='Superposition')
plot_comparison(results1, 'Demo 1: Superposition |+> State',
                expected={'0': 2048, '1': 2048})

# Statistics
sim_c = results1['simulator']
total = sum(sim_c.values())
print(f"\nSimulator: P(0)={sim_c.get('0',0)/total:.4f}  P(1)={sim_c.get('1',0)/total:.4f}")
if 'hardware' in results1:
    hw_c = results1['hardware']
    total_hw = sum(hw_c.values())
    print(f"Hardware:  P(0)={hw_c.get('0',0)/total_hw:.4f}  P(1)={hw_c.get('1',0)/total_hw:.4f}")

## Demo 2: Bell State — Entanglement

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

On a **perfect** computer: only $|00\rangle$ and $|11\rangle$ appear.  
On **real hardware**: $|01\rangle$ and $|10\rangle$ leak in due to gate errors.

In [ ]:
# Demo 2: Bell State

qc2 = QuantumCircuit(2, 2)
qc2.h(0)
qc2.cx(0, 1)
qc2.measure([0, 1], [0, 1])

print('Circuit:')
print(qc2.draw(output='text'))
print()

results2 = run_circuit(qc2, shots=4096, label='Bell State')
plot_comparison(results2, 'Demo 2: Bell State |Phi+>',
                expected={'00': 2048, '01': 0, '10': 0, '11': 2048})

# Fidelity estimation
sim_c = results2['simulator']
total = sum(sim_c.values())
fidelity_sim = (sim_c.get('00', 0) + sim_c.get('11', 0)) / total
print(f'\nBell state fidelity (simulator): {fidelity_sim:.4f}')
if 'hardware' in results2:
    hw_c = results2['hardware']
    total_hw = sum(hw_c.values())
    fidelity_hw = (hw_c.get('00', 0) + hw_c.get('11', 0)) / total_hw
    print(f'Bell state fidelity (hardware):  {fidelity_hw:.4f}')
    print(f'Error states leaked: {(1-fidelity_hw)*100:.1f}%')

## Demo 3: GHZ State — Multi-Qubit Entanglement

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$

GHZ states are extremely fragile — they reveal how noise scales with qubit count.

In [ ]:
# Demo 3: GHZ State (3 qubits)

n_ghz = 3
qc3 = QuantumCircuit(n_ghz, n_ghz)
qc3.h(0)
for i in range(1, n_ghz):
    qc3.cx(0, i)
qc3.measure(range(n_ghz), range(n_ghz))

print(f'{n_ghz}-qubit GHZ Circuit:')
print(qc3.draw(output='text'))
print()

results3 = run_circuit(qc3, shots=4096, label=f'{n_ghz}-qubit GHZ')
expected3 = {'0'*n_ghz: 2048, '1'*n_ghz: 2048}
plot_comparison(results3, f'Demo 3: {n_ghz}-Qubit GHZ State', expected=expected3)

# GHZ fidelity
sim_c = results3['simulator']
total = sum(sim_c.values())
f_sim = (sim_c.get('0'*n_ghz, 0) + sim_c.get('1'*n_ghz, 0)) / total
print(f'\nGHZ fidelity (simulator): {f_sim:.4f}')
if 'hardware' in results3:
    hw_c = results3['hardware']
    total_hw = sum(hw_c.values())
    f_hw = (hw_c.get('0'*n_ghz, 0) + hw_c.get('1'*n_ghz, 0)) / total_hw
    print(f'GHZ fidelity (hardware):  {f_hw:.4f}')

## Demo 4: Quantum Random Number Generator

Generate truly random numbers from quantum hardware — not pseudo-random!

In [ ]:
# Demo 4: QRNG — 4 qubits = 4 random bits per shot

n_qrng = 4
qc4 = QuantumCircuit(n_qrng, n_qrng)
for i in range(n_qrng):
    qc4.h(i)
qc4.measure(range(n_qrng), range(n_qrng))

print('QRNG Circuit:')
print(qc4.draw(output='text'))
print()

results4 = run_circuit(qc4, shots=4096, label='QRNG')

# Analyze uniformity
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for idx, (source, title, color) in enumerate([
    ('simulator', 'Simulator QRNG', '#2196F3'),
    ('hardware', f'Hardware QRNG ({backend.name if backend else "N/A"})', '#FF5722')
]):
    if source not in results4:
        if idx == 1:
            axes[1].text(0.5, 0.5, 'No hardware available\nPaste IBM token above!',
                        transform=axes[1].transAxes, ha='center', va='center', fontsize=14)
            axes[1].set_title(title, fontsize=13); axes[1].axis('off')
        continue
    
    counts = results4[source]
    all_states = [format(i, f'0{n_qrng}b') for i in range(2**n_qrng)]
    probs = [counts.get(s, 0)/sum(counts.values()) for s in all_states]
    
    axes[idx].bar(range(len(all_states)), probs, color=color, alpha=0.7, edgecolor='black')
    axes[idx].axhline(y=1/2**n_qrng, color='red', linestyle='--', linewidth=2, 
                     label=f'Uniform ({1/2**n_qrng:.3f})')
    axes[idx].set_xlabel('Outcome'); axes[idx].set_ylabel('Probability')
    axes[idx].set_title(title, fontsize=13, fontweight='bold')
    axes[idx].legend(); axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].set_xticks(range(0, 2**n_qrng, 2))
    axes[idx].set_xticklabels([all_states[i] for i in range(0, 2**n_qrng, 2)], rotation=45, fontsize=8)

plt.suptitle('Demo 4: Quantum Random Number Generator', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Extract random numbers
sim_samples = list(results4['simulator'].keys())[:8]
print(f'\nSample quantum random numbers: {sim_samples}')
print(f'As integers: {[int(s, 2) for s in sim_samples]}')

## Demo 5: Grover's Search (2-qubit)

Search for the marked state $|11\rangle$ among 4 possibilities.  
Classical: needs ~2 queries on average. Quantum: needs just **1**.

$$\text{Oracle}: |11\rangle \to -|11\rangle$$

In [ ]:
# Demo 5: Grover's Search — find |11>

qc5 = QuantumCircuit(2, 2)

# Initialize superposition
qc5.h([0, 1])

# Oracle: mark |11> with phase flip
qc5.cz(0, 1)

# Diffusion operator
qc5.h([0, 1])
qc5.z([0, 1])
qc5.cz(0, 1)
qc5.h([0, 1])

qc5.measure([0, 1], [0, 1])

print("Grover's Circuit (searching for |11>):")
print(qc5.draw(output='text'))
print()

results5 = run_circuit(qc5, shots=4096, label="Grover's Search")
plot_comparison(results5, "Demo 5: Grover's Search for |11>",
                expected={'00': 0, '01': 0, '10': 0, '11': 4096})

sim_c = results5['simulator']
total = sum(sim_c.values())
success_rate = sim_c.get('11', 0) / total
print(f'\nGrover success rate (simulator): {success_rate*100:.1f}%')
print(f'Classical random guess: 25%')
print(f'Quantum speedup: {success_rate/0.25:.1f}x')
if 'hardware' in results5:
    hw_c = results5['hardware']
    total_hw = sum(hw_c.values())
    hw_success = hw_c.get('11', 0) / total_hw
    print(f'Grover success rate (hardware): {hw_success*100:.1f}%')

## Demo 6: Noise Analysis — Simulator vs Hardware

Quantify the difference between ideal and real quantum computation.

In [ ]:
# Demo 6: Comprehensive Noise Analysis

# Run circuits of increasing depth and compare fidelity
depths = [1, 2, 3, 5, 8]
sim_fidelities = []
hw_fidelities = []

print('Running depth analysis...')
for depth in depths:
    qc_depth = QuantumCircuit(2, 2)
    qc_depth.h(0)
    qc_depth.cx(0, 1)
    # Add identity-like layers (H-H cancels but adds gate errors on hardware)
    for _ in range(depth - 1):
        qc_depth.barrier()
        qc_depth.h(0); qc_depth.h(0)  # H*H = I (noiseless), but adds noise on hardware
        qc_depth.cx(0, 1); qc_depth.cx(0, 1)  # CX*CX = I
    qc_depth.measure([0, 1], [0, 1])
    
    res = run_circuit(qc_depth, shots=4096, label=f'Depth-{depth}')
    
    # Simulator fidelity
    sc = res['simulator']
    st = sum(sc.values())
    sim_fidelities.append((sc.get('00', 0) + sc.get('11', 0)) / st)
    
    # Hardware fidelity
    if 'hardware' in res:
        hc = res['hardware']
        ht = sum(hc.values())
        hw_fidelities.append((hc.get('00', 0) + hc.get('11', 0)) / ht)
    
    print(f'  Depth {depth}: sim_fidelity={sim_fidelities[-1]:.4f}', end='')
    if hw_fidelities:
        print(f'  hw_fidelity={hw_fidelities[-1]:.4f}', end='')
    print()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(depths, sim_fidelities, 'b-o', linewidth=2.5, markersize=10, label='Simulator')
if hw_fidelities:
    axes[0].plot(depths, hw_fidelities, 'r-s', linewidth=2.5, markersize=10, label=f'Hardware ({backend.name})')
axes[0].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Circuit Depth (gate layers)', fontsize=12)
axes[0].set_ylabel('Bell State Fidelity', fontsize=12)
axes[0].set_title('Fidelity vs Circuit Depth', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.05)

# Error rates
sim_errors = [1 - f for f in sim_fidelities]
axes[1].semilogy(depths, [max(e, 1e-6) for e in sim_errors], 'b-o', linewidth=2.5, markersize=10, label='Simulator')
if hw_fidelities:
    hw_errors = [1 - f for f in hw_fidelities]
    axes[1].semilogy(depths, [max(e, 1e-6) for e in hw_errors], 'r-s', linewidth=2.5, markersize=10, label='Hardware')
axes[1].set_xlabel('Circuit Depth', fontsize=12)
axes[1].set_ylabel('Error Rate (1 - Fidelity)', fontsize=12)
axes[1].set_title('Error Scaling with Depth', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

plt.suptitle('Demo 6: Noise Analysis', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## Summary Dashboard

In [ ]:
# Summary Dashboard

print('=' * 65)
print('         IBM QUANTUM HARDWARE DEMONSTRATION SUMMARY')
print('=' * 65)
print()
print(f'  Mode:     {"REAL HARDWARE (" + backend.name + ")" if USE_REAL_HARDWARE else "SIMULATOR ONLY"}')
if USE_REAL_HARDWARE:
    print(f'  Backend:  {backend.name} ({backend.num_qubits} qubits)')
print(f'  Shots:    4,096 per experiment')
print(f'  Demos:    6 quantum circuits executed')
print()
print('  Results:')
print('  ' + '-' * 55)
print(f'  {"Demo":<35} {"Sim":>8} {"Hardware":>10}')
print('  ' + '-' * 55)

demos = [
    ('1. Superposition (H gate)', results1),
    ('2. Bell State (H + CNOT)', results2),
    (f'3. GHZ State ({n_ghz} qubits)', results3),
    ('4. QRNG (4 qubits)', results4),
    ('5. Grover Search (find |11>)', results5),
]

for name, res in demos:
    sim_ok = 'Pass' if res.get('simulator') else 'N/A'
    hw_ok = 'Pass' if res.get('hardware') else 'N/A'
    print(f'  {name:<35} {sim_ok:>8} {hw_ok:>10}')

print('  ' + '-' * 55)
print()
if not USE_REAL_HARDWARE:
    print('  To enable real hardware:')
    print('  1. Get token: https://quantum.ibm.com/account')
    print('  2. Paste in IBM_TOKEN variable above')
    print('  3. Re-run all cells')
print()
print('=' * 65)

## How to Read the Results

### Simulator vs Hardware Differences

| Effect | Cause | Example |
|--------|-------|--------|
| **Bit flip errors** | Gate imperfections | $|00\rangle$ appears as $|01\rangle$ |
| **Readout errors** | Measurement noise | $|0\rangle$ misread as $|1\rangle$ |
| **Decoherence** | T1/T2 decay | State drifts during computation |
| **Crosstalk** | Qubit coupling | Neighboring qubits interfere |

### Fidelity Interpretation

| Fidelity | Quality |
|----------|--------|
| > 0.99 | Excellent (few errors) |
| 0.95-0.99 | Good (typical for 1-2 qubit circuits) |
| 0.80-0.95 | Fair (deeper circuits) |
| < 0.80 | Poor (too many gates for current hardware) |

### References

1. IBM Quantum: https://quantum.ibm.com
2. Qiskit Documentation: https://docs.quantum.ibm.com
3. Qiskit Textbook: https://learning.quantum.ibm.com